# 🔬 Notebook 2 — Cosine · Euclidean · Dot Product Similarity: Deep Comparison

**What you'll learn**
- Implement all three similarity measures from scratch *and* via NumPy
- Understand *when each metric wins* with mathematical intuition
- Run all three against every domain and compare results side-by-side
- Score & rank queries to see which metric gives the best results
- Visualise score distributions, retrieval overlaps, and a decision framework

> **Pre-requisite**: Run Notebook 1 first (generates `embedding_outputs/` files)


## 1 · Setup & Load Embeddings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
np.random.seed(42)

OUT = Path("embedding_outputs")
assert OUT.exists(), "Run Notebook 1 first!"

df_all = pd.read_csv(OUT / "all_datasets.csv")
X = np.load(OUT / "embeddings_lsa.npy")   # L2-normalised LSA, (N, 256)
X_dense = np.load(OUT / "embeddings_dense.npy")  # unit-norm dense, (N, 384)

print(f"Dataset: {len(df_all)} records")
print(f"LSA embeddings: {X.shape}")
print(f"Dense embeddings: {X_dense.shape}")
print(f"Domains: {df_all.domain.unique().tolist()}")


## 2 · Three Similarity Metrics — Math & Implementation

### 2.1 Formulas

| Metric | Formula | Range | Best when |
|--------|---------|-------|-----------|
| **Cosine** | cos(θ) = (A·B) / (‖A‖ ‖B‖) | [−1, 1] | Direction matters, magnitude irrelevant |
| **Euclidean** | d = √Σ(aᵢ − bᵢ)² → sim = 1/(1+d) | (0, 1] | Magnitude AND direction matter |
| **Dot Product** | A·B = Σ aᵢbᵢ | (−∞, +∞) | Vectors already unit-normalised (= cosine) |

> **Key insight**: For L2-normalised vectors, dot product ≡ cosine similarity.  
> Raw (non-normalised) embeddings: dot product biases towards longer/louder vectors.


In [ ]:
# ─────────────────────────────────────────────────
# From-scratch implementations
# ─────────────────────────────────────────────────

def cosine_similarity_scratch(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def euclidean_similarity_scratch(a: np.ndarray, b: np.ndarray) -> float:
    """Convert Euclidean distance to similarity score in [0, 1]."""
    dist = np.linalg.norm(a - b)
    return 1.0 / (1.0 + dist)

def dot_product_scratch(a: np.ndarray, b: np.ndarray) -> float:
    """Raw dot product — scale depends on vector magnitudes."""
    return float(np.dot(a, b))

# Batch versions (matrix × vector)
def cosine_batch(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Cosine similarity: query (D,) vs all rows in matrix (N, D)."""
    q_norm = query / (np.linalg.norm(query) + 1e-10)
    m_norm = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-10)
    return m_norm @ q_norm

def euclidean_batch(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Euclidean similarity (=1/(1+dist)) — query vs all rows."""
    diffs = matrix - query[np.newaxis, :]
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    return 1.0 / (1.0 + dists)

def dot_product_batch(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Raw dot product — query vs all rows."""
    return matrix @ query

# Verify on a small example
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print("Verification on a=[1,2,3], b=[4,5,6]")
print(f"  cosine   : {cosine_similarity_scratch(a,b):.6f}")
print(f"  euclidean: {euclidean_similarity_scratch(a,b):.6f}")
print(f"  dot prod : {dot_product_scratch(a,b):.6f}")
print()
# Unit-norm case
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
print("For unit-normalised vectors:")
print(f"  cosine   : {cosine_similarity_scratch(a_n, b_n):.6f}")
print(f"  dot prod : {dot_product_scratch(a_n, b_n):.6f}  ← should match cosine")


## 3 · Side-by-Side Retrieval Comparison

We'll run the **same query** through all three metrics and compare top-5 results.


In [ ]:
def retrieve_top_k(query_idx: int, X: np.ndarray, df: pd.DataFrame,
                   k: int = 5, exclude_self: bool = True):
    """Return top-k results for all three metrics from a single query."""
    query = X[query_idx]
    
    cos_scores = cosine_batch(query, X)
    euc_scores = euclidean_batch(query, X)
    dot_scores = dot_product_batch(query, X)
    
    if exclude_self:
        cos_scores[query_idx] = -np.inf
        euc_scores[query_idx] = -np.inf
        dot_scores[query_idx] = -np.inf
    
    results = {}
    for name, scores in [("cosine", cos_scores),
                          ("euclidean", euc_scores),
                          ("dot_product", dot_scores)]:
        top_idx = np.argsort(scores)[::-1][:k]
        results[name] = pd.DataFrame({
            "rank": range(1, k+1),
            "score": scores[top_idx],
            "domain": df.iloc[top_idx].domain.values,
            "category": df.iloc[top_idx].category.values,
            "text": df.iloc[top_idx].text.str[:90].values
        })
    return results

def print_comparison(query_idx: int, X: np.ndarray, df: pd.DataFrame, k: int = 5):
    query_row = df.iloc[query_idx]
    print(f"\n{'='*80}")
    print(f"QUERY [{query_row.domain.upper()} › {query_row.category}]")
    print(f"  "{query_row.text[:120]}"")
    print(f"{'='*80}")
    
    results = retrieve_top_k(query_idx, X, df, k)
    
    for metric, res in results.items():
        print(f"\n  ── {metric.upper()} ──")
        for _, row in res.iterrows():
            match = "✅" if row.category == query_row.category else "❌"
            print(f"    {match} #{row['rank']:1d} [{row.domain:8s}›{row.category:15s}] "
                  f"score={row.score:7.4f}  {row.text[:65]}")

# ── Clinical query ──
clinical_idx = df_all[df_all.category == "cardiology"].index[0]
print_comparison(clinical_idx, X, df_all)

# ── Failure query ──
failure_idx = df_all[df_all.category == "electrical"].index[0]
print_comparison(failure_idx, X, df_all)


In [ ]:
# ── Support ticket query ──
support_idx = df_all[df_all.category == "billing"].index[0]
print_comparison(support_idx, X, df_all)

# ── Research query ──
research_idx = df_all[df_all.category == "machine_learning"].index[0]
print_comparison(research_idx, X, df_all)


## 4 · Quantitative Accuracy Benchmark — All Domains × All Metrics

We measure **Precision@K**: what fraction of the top-K results share the query's category.


In [ ]:
from tqdm.auto import tqdm

def precision_at_k(query_idx, scores, y_true, k=10):
    """Fraction of top-k neighbours with same label as query."""
    scores = scores.copy()
    scores[query_idx] = -np.inf
    top_k = np.argsort(scores)[::-1][:k]
    return (y_true[top_k] == y_true[query_idx]).mean()

def benchmark_domain(domain, X, df, ks=(1, 5, 10, 20)):
    mask = (df.domain == domain).values
    idx_list = np.where(mask)[0]
    X_d = X          # use full matrix but restrict queries to domain
    labels = df.category.values
    
    rows = []
    # Sample up to 150 queries to keep runtime manageable
    sample = np.random.choice(idx_list, min(150, len(idx_list)), replace=False)
    
    for qidx in sample:
        q = X_d[qidx]
        cos_s = cosine_batch(q, X_d)
        euc_s = euclidean_batch(q, X_d)
        dot_s = dot_product_batch(q, X_d)
        
        for metric_name, scores in [("cosine", cos_s),
                                     ("euclidean", euc_s),
                                     ("dot_product", dot_s)]:
            for k in ks:
                p = precision_at_k(qidx, scores, labels, k)
                rows.append({"domain": domain, "metric": metric_name, "k": k, "precision": p})
    
    return pd.DataFrame(rows)

print("Benchmarking all domains × metrics (may take ~30s) …")
bench_frames = []
for domain in df_all.domain.unique():
    print(f"  {domain} …", end=" ", flush=True)
    bench_frames.append(benchmark_domain(domain, X, df_all))
    print("done")

df_bench = pd.concat(bench_frames, ignore_index=True)
df_summary = df_bench.groupby(["domain","metric","k"])["precision"].mean().reset_index()
print("\n✅  Benchmark complete")
df_summary.head(15)


In [ ]:
# ─────────────────────────────────────────────────
# Heatmap: P@K for each domain × metric
# ─────────────────────────────────────────────────
k_to_show = 10
pivot = df_summary[df_summary.k == k_to_show].pivot_table(
    index="domain", columns="metric", values="precision")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap
sns.heatmap(pivot, annot=True, fmt=".2%", cmap="YlGn", ax=axes[0],
            linewidths=0.5, cbar_kws={"label": f"Precision@{k_to_show}"})
axes[0].set_title(f"Precision@{k_to_show} — Domain × Metric", fontsize=12, fontweight="bold")
axes[0].set_xlabel(""); axes[0].set_ylabel("")

# Bar comparison per metric
domains = df_all.domain.unique()
x = np.arange(len(domains))
width = 0.25
colors = {"cosine": "#2A9D8F", "euclidean": "#E76F51", "dot_product": "#457B9D"}
for i, metric in enumerate(["cosine", "euclidean", "dot_product"]):
    vals = [pivot.loc[d, metric] if d in pivot.index else 0 for d in domains]
    axes[1].bar(x + i*width, vals, width, label=metric, color=colors[metric], alpha=0.85)

axes[1].set_xticks(x + width); axes[1].set_xticklabels(domains, rotation=20)
axes[1].set_ylabel(f"Precision@{k_to_show}")
axes[1].set_title(f"Metric Comparison across Domains (P@{k_to_show})", fontsize=12, fontweight="bold")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT / "metric_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# P@K curves — how does performance degrade as K grows?
# ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = {"cosine": "#2A9D8F", "euclidean": "#E76F51", "dot_product": "#457B9D"}

for i, domain in enumerate(df_all.domain.unique()):
    ax = axes[i]
    sub = df_summary[df_summary.domain == domain]
    for metric in ["cosine", "euclidean", "dot_product"]:
        m_data = sub[sub.metric == metric].sort_values("k")
        ax.plot(m_data.k, m_data.precision, marker="o", label=metric,
                color=colors[metric], linewidth=2, markersize=5)
    ax.set_title(f"Domain: {domain.upper()}", fontsize=11, fontweight="bold")
    ax.set_xlabel("K"); ax.set_ylabel("Precision@K")
    ax.legend(fontsize=8); ax.grid(alpha=0.4)
    ax.set_ylim(0, 1)

axes[-1].set_visible(False)
plt.suptitle("Precision@K Curves — All Domains × All Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "precision_at_k_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 5 · Score Distribution Analysis

Understanding score distributions helps you **set retrieval thresholds**.  
A well-separated distribution means you can confidently distinguish relevant vs irrelevant.


In [ ]:
# Sample 1000 pairs: same-category vs different-category
def sample_pairs(X, labels, n=1000):
    n = min(n, len(X))
    idx_a = np.random.randint(0, len(X), n)
    idx_b = np.random.randint(0, len(X), n)
    same = labels[idx_a] == labels[idx_b]
    return idx_a, idx_b, same

labels = df_all.category.values
idx_a, idx_b, same = sample_pairs(X, labels, 2000)

cos_scores   = (X[idx_a] * X[idx_b]).sum(axis=1)
euc_scores   = 1 / (1 + np.sqrt(((X[idx_a] - X[idx_b])**2).sum(axis=1)))
dot_scores   = (X[idx_a] * X[idx_b]).sum(axis=1)  # same as cosine for unit vecs

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_data = [
    ("Cosine Similarity", cos_scores, "#2A9D8F"),
    ("Euclidean Similarity", euc_scores, "#E76F51"),
    ("Dot Product", dot_scores, "#457B9D"),
]

for ax, (name, scores, color) in zip(axes, metric_data):
    ax.hist(scores[same], bins=60, alpha=0.65, label="Same category", color=color, density=True)
    ax.hist(scores[~same], bins=60, alpha=0.65, label="Diff category", color="grey", density=True)
    ax.set_title(f"{name} Distribution", fontsize=11, fontweight="bold")
    ax.set_xlabel("Score"); ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Overlap coefficient
    from scipy.stats import gaussian_kde
    try:
        kde_same = gaussian_kde(scores[same])
        kde_diff = gaussian_kde(scores[~same])
        x_range = np.linspace(scores.min(), scores.max(), 300)
        overlap = np.minimum(kde_same(x_range), kde_diff(x_range)).sum() / kde_same(x_range).sum()
        ax.set_xlabel(f"Score (overlap≈{overlap:.1%})")
    except:
        pass

plt.suptitle("Score Distributions: Same vs Different Category Pairs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("💡 Less overlap = better discrimination. Cosine usually wins on normalised embeddings.")


## 6 · Retrieval Overlap — Do Metrics Agree?

How often do cosine, euclidean, and dot product return the **same top-K results**?  
High overlap → metrics are interchangeable; low overlap → choice matters!


In [ ]:
def top_k_set(scores, query_idx, k=10):
    s = scores.copy(); s[query_idx] = -np.inf
    return set(np.argsort(s)[::-1][:k])

def jaccard(set_a, set_b):
    if not set_a and not set_b: return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

# Sample 200 queries
sample_idx = np.random.choice(len(df_all), 200, replace=False)
overlap_cos_euc, overlap_cos_dot, overlap_euc_dot = [], [], []

for qidx in sample_idx:
    q = X[qidx]
    cos_s = cosine_batch(q, X)
    euc_s = euclidean_batch(q, X)
    dot_s = dot_product_batch(q, X)
    
    k = 10
    top_cos = top_k_set(cos_s, qidx, k)
    top_euc = top_k_set(euc_s, qidx, k)
    top_dot = top_k_set(dot_s, qidx, k)
    
    overlap_cos_euc.append(jaccard(top_cos, top_euc))
    overlap_cos_dot.append(jaccard(top_cos, top_dot))
    overlap_euc_dot.append(jaccard(top_euc, top_dot))

print("Jaccard Overlap of Top-10 Retrieved Sets (200 sampled queries):")
print(f"  Cosine ∩ Euclidean  : {np.mean(overlap_cos_euc):.2%}  ± {np.std(overlap_cos_euc):.2%}")
print(f"  Cosine ∩ Dot Product: {np.mean(overlap_cos_dot):.2%}  ± {np.std(overlap_cos_dot):.2%}")
print(f"  Euclidean ∩ Dot Prod: {np.mean(overlap_euc_dot):.2%}  ± {np.std(overlap_euc_dot):.2%}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(overlap_cos_euc, bins=30, alpha=0.7, label="Cosine ∩ Euclidean", color="#2A9D8F")
ax.hist(overlap_cos_dot, bins=30, alpha=0.7, label="Cosine ∩ Dot Product", color="#457B9D")
ax.hist(overlap_euc_dot, bins=30, alpha=0.7, label="Euclidean ∩ Dot Prod", color="#E76F51")
ax.axvline(np.mean(overlap_cos_euc), color="#2A9D8F", linestyle="--", linewidth=2)
ax.axvline(np.mean(overlap_cos_dot), color="#457B9D", linestyle="--", linewidth=2)
ax.axvline(np.mean(overlap_euc_dot), color="#E76F51", linestyle="--", linewidth=2)
ax.set_xlabel("Jaccard Overlap (Top-10)"); ax.set_ylabel("Count")
ax.set_title("Retrieval Set Overlap between Metrics", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "retrieval_overlap.png", dpi=150, bbox_inches="tight")
plt.show()


## 7 · Which Metric Is Best for Each Dataset?

A decision framework based on our benchmark results.


In [ ]:
# Winner per domain at P@10
winner_rows = []
for domain in df_all.domain.unique():
    sub = df_summary[(df_summary.domain == domain) & (df_summary.k == 10)]
    best = sub.loc[sub.precision.idxmax()]
    winner_rows.append({
        "domain": domain,
        "best_metric": best.metric,
        "precision@10": best.precision,
        "recommendation": {
            "clinical": "Cosine — clinical notes vary in length; direction matters more than magnitude",
            "reviews": "Cosine or Dot — sentiment directionality maps well to cosine angle",
            "failures": "Euclidean — failure embeddings encode severity distance; magnitude relevant",
            "support": "Cosine — tickets have very different lengths; normalisation essential",
            "research": "Cosine — abstracts are uniform length; semantic angle is the key signal",
        }.get(domain, "Cosine")
    })

df_recs = pd.DataFrame(winner_rows)
print("\n📋 METRIC RECOMMENDATION SUMMARY")
print("="*70)
for _, row in df_recs.iterrows():
    print(f"  {row.domain:12s} | Best: {row.best_metric:12s} | P@10: {row['precision@10']:.1%}")
    print(f"              → {row.recommendation}")
    print()

# Visualise
fig, ax = plt.subplots(figsize=(12, 5))
metric_colors = {"cosine": "#2A9D8F", "euclidean": "#E76F51", "dot_product": "#457B9D"}
colors = [metric_colors.get(m, "grey") for m in df_recs.best_metric]
bars = ax.barh(df_recs.domain, df_recs["precision@10"], color=colors, edgecolor="white", height=0.5)
ax.set_xlabel("Precision@10"); ax.set_title("Best Metric per Domain (P@10)", fontsize=12, fontweight="bold")
for bar, (_, row) in zip(bars, df_recs.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            row.best_metric, va="center", fontsize=9, color=metric_colors.get(row.best_metric, "grey"))
ax.grid(axis="x", alpha=0.4); ax.set_xlim(0, 1.15)

from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=m) for m, c in metric_colors.items()]
ax.legend(handles=legend, loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "metric_recommendations.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────
# Final decision flowchart (text-based)
# ─────────────────────────────────────────────────
flowchart = """
╔══════════════════════════════════════════════════════════════════════╗
║           SIMILARITY METRIC DECISION FRAMEWORK                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Are your vectors L2-normalised (unit length)?                     ║
║    YES → Cosine ≡ Dot Product  (pick Dot Product — it's faster)   ║
║    NO  ↓                                                           ║
║                                                                    ║
║  Does vector MAGNITUDE carry semantic information?                 ║
║    YES (e.g., TF-IDF without normalisation, count vectors)         ║
║        → Euclidean distance                                        ║
║    NO  (e.g., sentence embeddings, BERT, all equal importance)     ║
║        → Cosine similarity                                         ║
║                                                                    ║
║  Dataset-specific guidance:                                        ║
║    Clinical notes      → Cosine  (length varies, direction matters)║
║    Product reviews     → Cosine  (sentiment direction key)         ║
║    Equipment failures  → Euclidean (severity magnitude matters)    ║
║    Support tickets     → Cosine  (short vs long tickets)           ║
║    Research abstracts  → Cosine  (semantic purity)                 ║
║                                                                    ║
║  ANN index choice:                                                 ║
║    Cosine/Dot  → HNSW with inner_product or cosine space          ║
║    Euclidean   → HNSW with L2 space                                ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(flowchart)


In [ ]:
print("="*60)
print("NOTEBOOK 2 COMPLETE")
print("→ Key takeaways saved in:", OUT)
for f in ["metric_benchmark.png", "precision_at_k_curves.png",
          "score_distributions.png", "retrieval_overlap.png",
          "metric_recommendations.png"]:
    print(f"  {f}")
print("\n→ Proceed to Notebook 3: ANN Search & Retrieval Systems")
